# Симулятор диффузии/теплопроводности 2D

Данный код моделирует двумерную линейную модель диффузии/теплопроводности, которая подчиняется уравнению в частных производных вида

$$U_t = a \left( U_{xx} + U_{yy} \right)$$

где:
- $a$ — постоянный коэффициент теплопроводности/диффузии
- $U$ — физическая величина (температура, концентрация и т.п.)

Для решения используется метод конечного объёма на прямоугольной сетке.  
Два метода интегрирования по времени:

1. **Явная схема (explicit)** — стандартный явный метод Эйлера. Ограничение по шагу:
$$\Delta t \leq \frac{\Delta x^2 \Delta y^2}{2\,a\,(\Delta x^2 + \Delta y^2)}$$

2. **Метод RKL2 (super time-stepping)** — метод Рунге-Кутты-Лежандра 2-го порядка.  
   Позволяет делать супер-шаги размером $\Delta t_{\rm super} = \Delta t_{\rm expl}\,(s^2+s-2)/4$,  
   что даёт ускорение в $\sim s^2/4$ раз по сравнению с явной схемой при $s$ стадиях.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
import sys

In [ ]:
'''
###############################################################################
#
#  Двумерная прямоугольная сетка с фиктивными ячейками (ghost cells).
#
#  Реальные ячейки: U[1:Nx+1, 1:Ny+1]
#  Фиктивные ячейки: строки/столбцы с индексами 0 и Nx+1 (Ny+1)
#
#        j=0  j=1  ...  j=Ny  j=Ny+1
#  i=0   [G]  [G]  ...  [G]   [G]
#  i=1   [G]  [R]  ...  [R]   [G]
#  ...   [G]  [R]  ...  [R]   [G]
#  i=Nx  [G]  [R]  ...  [R]   [G]
#  i=Nx+1[G]  [G]  ...  [G]   [G]
#  G = ghost cell, R = real cell
#
###############################################################################
'''

def grid_setup(xmin, xmax, ymin, ymax, Nx, Ny):
    """
    Создаёт двумерную равномерную сетку с фиктивными ячейками.

    Возвращает:
        dx, dy   -- шаги сетки по x и y
        cx, cy   -- 1D массивы координат центров ячеек (включая фиктивные)
        CX, CY   -- 2D meshgrid центров ячеек (включая фиктивные)
    """
    dx = (xmax - xmin) / Nx
    dy = (ymax - ymin) / Ny

    # координаты центров ячеек (включая одну фиктивную с каждой стороны)
    cx = np.linspace(xmin - 0.5*dx, xmax + 0.5*dx, Nx + 2, dtype=np.double)
    cy = np.linspace(ymin - 0.5*dy, ymax + 0.5*dy, Ny + 2, dtype=np.double)

    CX, CY = np.meshgrid(cx, cy, indexing='ij')  # shape: (Nx+2, Ny+2)

    return dx, dy, cx, cy, CX, CY

In [ ]:
'''
###############################################################################
#
#  Начальные условия.
#
#  setup = 'disc'   -- прямоугольный "горячий" квадрат в центре области
#  setup = 'smooth' -- гауссов "пакет" в центре области
#  setup = 'cross'  -- крест из двух гауссовых пакетов
#
###############################################################################
'''

def initial_solution(xmin, xmax, ymin, ymax, CX, CY, dx, dy, Nx, Ny, setup):
    """
    Задаёт начальное условие U0 (shape: Nx+2, Ny+2) и коэффициент a.
    """
    a = 1.0

    U0 = np.ones((Nx + 2, Ny + 2), dtype=np.double)

    x0 = 0.5 * (xmin + xmax)
    y0 = 0.5 * (ymin + ymax)

    if setup == 'disc':
        # горячий квадрат шириной 20% области в центре
        Lx = xmax - xmin
        Ly = ymax - ymin
        mask = ((np.abs(CX - x0) < 0.1*Lx) & (np.abs(CY - y0) < 0.1*Ly))
        U0[mask] = 2.0

    elif setup == 'smooth':
        # гауссов пакет
        delta_x = 5.0 * dx
        delta_y = 5.0 * dy
        U0 = np.exp(-((CX - x0)**2 / delta_x**2 + (CY - y0)**2 / delta_y**2))

    elif setup == 'cross':
        # два пересекающихся гауссовых пакета
        Lx = xmax - xmin
        Ly = ymax - ymin
        sigma = 0.08 * max(Lx, Ly)
        U0 = (np.exp(-((CX - x0)**2 + (CY - y0)**2) / sigma**2) +
              np.exp(-((CX - x0)**2) / (0.5*sigma)**2) * np.exp(-((CY - y0)**2) / (4*sigma)**2))

    else:
        sys.exit("error: choose setup from 'disc', 'smooth', 'cross'")

    return U0, a

In [ ]:
'''
###############################################################################
#
#  Пространственный оператор: дискретный лапласиан методом конечных разностей
#  второго порядка точности.
#
#  L(U)[i,j] = a * (U[i+1,j] - 2*U[i,j] + U[i-1,j]) / dx^2
#            + a * (U[i,j+1] - 2*U[i,j] + U[i,j-1]) / dy^2
#
#  Входной массив U имеет размер (Nx+2, Ny+2); результат -- размер (Nx, Ny)
#  (только внутренние ячейки).
#
###############################################################################
'''

def spatial_operator(U, a, dx, dy):
    """
    Вычисляет a * Laplacian(U) на внутренних ячейках.
    Возвращает массив формы (Nx, Ny).
    """
    LU = (a * (U[2:, 1:-1] - 2.0*U[1:-1, 1:-1] + U[:-2, 1:-1]) / dx**2 +
          a * (U[1:-1, 2:] - 2.0*U[1:-1, 1:-1] + U[1:-1, :-2]) / dy**2)
    return LU


def apply_bc(U, Nx, Ny):
    """
    Граничные условия Неймана (свободная/нулевой поток):
        dU/dn = 0 на всех границах.
    Реализовано зеркальным отражением в фиктивные ячейки.
    """
    U[0, :]    = U[1, :]      # левая граница
    U[Nx+1, :] = U[Nx, :]    # правая граница
    U[:, 0]    = U[:, 1]      # нижняя граница
    U[:, Ny+1] = U[:, Ny]    # верхняя граница
    return U

In [ ]:
'''
###############################################################################
#
#  Явная схема (метод Эйлера по времени).
#
#  U^{n+1} = U^n + dt * L(U^n)
#
#  Устойчивость требует:
#      dt <= dx^2 * dy^2 / (2 * a * (dx^2 + dy^2))
#
###############################################################################
'''

def explicit_step(U0, Nx, Ny, dx, dy, a, dt):
    """
    Один явный шаг по времени.

    Параметры:
        U0  : массив решения (Nx+2, Ny+2), включая фиктивные ячейки
        dt  : шаг по времени

    Возвращает:
        U1  : решение на следующем шаге (Nx+2, Ny+2)
    """
    U0 = apply_bc(U0, Nx, Ny)

    U1 = U0.copy()
    U1[1:-1, 1:-1] = U0[1:-1, 1:-1] + dt * spatial_operator(U0, a, dx, dy)

    return U1

In [ ]:
'''
###############################################################################
#
#  Метод RKL2 (Runge-Kutta-Legendre, 2nd order) -- Super Time Stepping.
#
#  Алгоритм: Meyer, Balsara & Aslam (2014), MNRAS 422, 2102
#  (также известен как STS-RKL2).
#
#  За один "супер-шаг" dt_super = dt_expl * (s^2 + s - 2)/4 делается s стадий.
#  Порядок точности по времени: 2.
#  Устойчивость: улучшена в ~s^2/4 раз относительно явной схемы.
#
###############################################################################
'''

def rkl2_coefs(s):
    """
    Вычисляет коэффициенты RKL2 для s стадий.

    Возвращает:
        b, mu, nu, gamma  -- массивы коэффициентов длины s+1
    """
    w1 = 4.0 / (s**2 + s - 2.0)

    b     = np.zeros(s + 1)
    mu    = np.zeros(s + 1)
    nu    = np.zeros(s + 1)
    gamma = np.zeros(s + 1)

    b[0] = 1.0 / 3.0
    b[1] = 1.0 / 3.0
    for j in range(2, s + 1):
        b[j]     = (j**2 + j - 2.0) / (2.0 * j * (j + 1))
        mu[j]    = (2.0*j - 1.0) / j * b[j] / b[j-1]
        nu[j]    = -(j - 1.0) / j * b[j] / b[j-2]
        gamma[j] = -(1.0 - b[j-1]) * mu[j] * w1

    return b, mu, nu, gamma


def rkl2_step(U0, Nx, Ny, dx, dy, a, dt, b, mu, nu, gamma, s):
    """
    Один супер-шаг по времени методом RKL2.

    Параметры:
        U0           : массив решения (Nx+2, Ny+2)
        dt           : размер супер-шага
        b, mu, nu,
        gamma        : коэффициенты RKL2 (из rkl2_coefs)
        s            : число стадий

    Возвращает:
        Y1  : решение после супер-шага (Nx+2, Ny+2)
    """
    w1 = 4.0 / (s**2 + s - 2.0)

    U0 = apply_bc(U0.copy(), Nx, Ny)

    # начальное приближение и оператор L(U^n)
    Y0  = U0.copy()
    LU0 = spatial_operator(U0, a, dx, dy)   # shape (Nx, Ny)

    # стадия j=1
    Y1 = Y0.copy()
    Y1[1:-1, 1:-1] = Y0[1:-1, 1:-1] + (w1 / 3.0) * dt * LU0

    # стадии j = 2 ... s
    for j in range(2, s + 1):
        Y1 = apply_bc(Y1, Nx, Ny)
        Y2 = Y1.copy()

        Y2[1:-1, 1:-1] = (mu[j]    * Y1[1:-1, 1:-1]
                        + nu[j]    * Y0[1:-1, 1:-1]
                        + w1*mu[j] * dt * spatial_operator(Y1, a, dx, dy)
                        + (1.0 - mu[j] - nu[j]) * U0[1:-1, 1:-1]
                        + gamma[j] * dt * LU0)
        Y0 = Y1
        Y1 = Y2

    return Y1

In [ ]:
'''
###############################################################################
#
#   Основной код программы
#
###############################################################################
'''

# --- параметры области ---
xmin, xmax = 0.0, 10.0
ymin, ymax = 0.0, 10.0

# --- разрешение сетки ---
Nx = 200
Ny = 200

# --- строим сетку ---
dx, dy, cx, cy, CX, CY = grid_setup(xmin, xmax, ymin, ymax, Nx, Ny)

# --- начальные условия ---
# доступные: 'disc', 'smooth', 'cross'
setup = 'smooth'
U0, a = initial_solution(xmin, xmax, ymin, ymax, CX, CY, dx, dy, Nx, Ny, setup)

# --- финальное время ---
phystime_fin = 2.0

# --- выбор солвера ---
# solver = 'expl'  -- явная схема
# solver = 'rkl2'  -- super time stepping RKL2
solver = 'rkl2'
S = 20  # число стадий RKL2 (только для solver='rkl2')

# --- предварительный расчёт коэффициентов RKL2 ---
b, mu, nu, gamma = rkl2_coefs(S)

# --- CFL-ограничение явной схемы ---
tcfl = 0.9 * (dx**2 * dy**2) / (2.0 * a * (dx**2 + dy**2))

print(f"Grid: {Nx}x{Ny}   dx={dx:.4f}   dy={dy:.4f}")
print(f"dt_expl (CFL) = {tcfl:.6f}")
if solver == 'rkl2':
    dt_super = tcfl * (S**2 + S - 2) / 4.0
    print(f"dt_super (RKL2, s={S}) = {dt_super:.6f}  (x{dt_super/tcfl:.1f} larger)")
print(f"Solver: {solver}")

# --- начальный интеграл (сохранение меры) ---
Integr0 = np.sum(U0[1:-1, 1:-1]) * dx * dy
print(f"Initial integral = {Integr0:.6f}")

# --- цикл по времени ---
phystime = 0.0
nts = 0
start_time = time.time()

while phystime < phystime_fin - 1e-14:

    if solver == 'expl':
        dt = min(tcfl, phystime_fin - phystime)
        U1 = explicit_step(U0, Nx, Ny, dx, dy, a, dt)

    elif solver == 'rkl2':
        dt = min(phystime_fin - phystime, tcfl * (S**2 + S - 2) / 4.0)
        U1 = rkl2_step(U0, Nx, Ny, dx, dy, a, dt, b, mu, nu, gamma, S)

    else:
        sys.exit("error: choose solver = 'expl' or 'rkl2'")

    U0[1:-1, 1:-1] = U1[1:-1, 1:-1]
    phystime += dt
    nts += 1

end_time = time.time()
print(f"\nElapsed time = {end_time - start_time:.3f} s")
print(f"Num timesteps = {nts}")

Integr1 = np.sum(U0[1:-1, 1:-1]) * dx * dy
print(f"Final integral  = {Integr1:.6f}  (change: {(Integr1-Integr0)/Integr0*100:.4f}%)")

In [ ]:
'''
###############################################################################
#
#   Визуализация результата
#
###############################################################################
'''

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- цветовая карта (2D) ---
ax = axes[0]
im = ax.imshow(
    U0[1:-1, 1:-1].T,          # транспонируем: ось x -- горизонталь
    origin='lower',
    extent=[xmin, xmax, ymin, ymax],
    cmap='hot',
    aspect='equal'
)
fig.colorbar(im, ax=ax, label='U')
ax.set_title(f'U(x,y) at t = {phystime:.3f}   [{solver}, Nx={Nx}, Ny={Ny}]')
ax.set_xlabel('x')
ax.set_ylabel('y')

# --- профиль по центральному сечению y = ymax/2 ---
ax2 = axes[1]
jmid = Ny // 2
ax2.plot(cx[1:-1], U0[1:-1, jmid + 1], label=f'y = {cy[jmid+1]:.2f}')
ax2.set_title(f'Central x-slice at y ≈ {cy[jmid+1]:.2f}')
ax2.set_xlabel('x')
ax2.set_ylabel('U')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
'''
###############################################################################
#
#   Сравнение явной схемы и RKL2 по времени работы при одинаковом dt_super
#
###############################################################################
'''

def run_simulation(solver_name, Nx, Ny, phystime_fin, S=10):
    dx, dy, cx, cy, CX, CY = grid_setup(xmin, xmax, ymin, ymax, Nx, Ny)
    U0, a = initial_solution(xmin, xmax, ymin, ymax, CX, CY, dx, dy, Nx, Ny, 'smooth')
    tcfl = 0.9 * (dx**2 * dy**2) / (2.0 * a * (dx**2 + dy**2))
    b_, mu_, nu_, gamma_ = rkl2_coefs(S)

    phystime = 0.0
    nts = 0
    t0 = time.time()

    while phystime < phystime_fin - 1e-14:
        if solver_name == 'expl':
            dt = min(tcfl, phystime_fin - phystime)
            U1 = explicit_step(U0, Nx, Ny, dx, dy, a, dt)
        else:
            dt = min(phystime_fin - phystime, tcfl * (S**2 + S - 2) / 4.0)
            U1 = rkl2_step(U0, Nx, Ny, dx, dy, a, dt, b_, mu_, nu_, gamma_, S)
        U0[1:-1, 1:-1] = U1[1:-1, 1:-1]
        phystime += dt
        nts += 1

    elapsed = time.time() - t0
    return elapsed, nts


print("Benchmarking explicit vs RKL2 (Nx=Ny=100, t_fin=0.5)")
print("-" * 55)
for s_val in [5, 10, 20]:
    t_expl, n_expl = run_simulation('expl', 100, 100, 0.5)
    t_rkl2, n_rkl2 = run_simulation('rkl2', 100, 100, 0.5, S=s_val)
    speedup = t_expl / t_rkl2
    print(f"  S={s_val:2d}: expl {n_expl:5d} steps {t_expl:.3f}s  |  "
          f"rkl2 {n_rkl2:4d} steps {t_rkl2:.3f}s  |  speedup x{speedup:.1f}")